# Q1 — Mạng tích chập từ đầu: Tích chập và gộp cực đại

Trong Assignment 1, mỗi ảnh được làm phẳng thành một vector trước khi đưa vào mạng kết
nối đầy đủ. Cách làm đó bỏ qua cấu trúc không gian của ảnh.

Mạng nơ-ron tích chập (**Convolutional Neural Network — CNN**) giữ lại cấu trúc hai chiều
và học các bộ lọc cục bộ (**filters/bộ lọc (kernel)s**) có thể phát hiện cạnh, họa tiết và các mẫu
hình ảnh phức tạp hơn.

Trong notebook này, bạn sẽ tự cài đặt các phép toán nền tảng của CNN bằng NumPy:

$$
\text{Convolution}
\rightarrow
\text{ReLU}
\rightarrow
\text{Max Pooling}.
$$

Mục tiêu là hiểu rõ:

- cách tính kích thước đầu ra của phép tích chập;
- vai trò của bước trượt (**stride**) và phần đệm (**padding**);
- cách một bộ lọc được dùng lặp lại trên toàn ảnh;
- truyền ngược qua convolution và max pooling;
- cách kiểm tra đạo hàm bằng PyTorch và vi phân số.

In [11]:
from pathlib import Path

assert Path("dl2").exists(), (
    "Không tìm thấy thư mục dl2/. "
    "Hãy chạy notebook từ thư mục gốc của Assignment 2."
)

print("Thư mục bài tập:", Path.cwd())

Thư mục bài tập: d:\Năm 3\học sâu DL\Assignment2_DeepLearning


In [12]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

from dl2.gradcheck import relative_error

np.random.seed(42)
torch.manual_seed(42)

## 1. Vì sao CNN phù hợp với ảnh?

Một ảnh màu CIFAR-10 có kích thước:

$$
3\times32\times32.
$$

Nếu làm phẳng, mỗi ảnh trở thành vector 3072 chiều. Một lớp kết nối đầy đủ phải học một
tham số riêng cho gần như mọi cặp vị trí đầu vào–đầu ra.

CNN dùng hai ý tưởng quan trọng:

### 1.1. Kết nối cục bộ

Một neuron chỉ nhìn một vùng nhỏ của ảnh, gọi là **trường tiếp nhận
(receptive field)**.

### 1.2. Chia sẻ tham số

Cùng một bộ lọc được trượt qua nhiều vị trí. Vì vậy một đặc trưng như cạnh dọc có thể
được phát hiện ở bất kỳ vị trí nào mà không cần học lại bộ lọc mới.

Đầu vào thường có dạng:

$$
X\in\mathbb{R}^{N\times C\times H\times W},
$$

trong đó:

- $N$: số mẫu trong lô;
- $C$: số kênh;
- $H,W$: chiều cao và chiều rộng.

Bộ lọc có dạng:

$$
W\in\mathbb{R}^{F\times C\times HH\times WW},
$$

với $F$ là số bộ lọc.

## 2. Kích thước đầu ra

Với stride $S$ và padding $P$:

$$
H_{\text{out}}
=
1+\frac{H+2P-HH}{S},
$$

$$
W_{\text{out}}
=
1+\frac{W+2P-WW}{S}.
$$

Các giá trị phải là số nguyên nếu muốn các cửa sổ quét khớp hoàn toàn.

### Ví dụ

Đầu vào $32\times32$, bộ lọc (kernel) $3\times3$, stride $1$, padding $1$:

$$
H_{\text{out}}
=
1+\frac{32+2-3}{1}
=32.
$$

Padding bằng 1 giúp giữ nguyên kích thước không gian.

### Câu hỏi 1 — Kích thước đầu ra

Không cần chạy code. Hãy tính kích thước không gian của đầu ra trong hai trường hợp:

1. $H=W=32$, bộ lọc (kernel) $5\times5$, stride $1$, padding $0$.
2. $H=W=32$, bộ lọc (kernel) $3\times3$, stride $2$, padding $1$.

Giải thích ngắn gọn vì sao stride lớn hơn 1 làm giảm độ phân giải.

### ✍️ Câu trả lời của bạn

<!-- Viết câu trả lời bên dưới. Không xóa ô này. -->
1. H_out = W_ out = 1 + (32 + 2x0 -5)/1 = 28
2. H_out = W_out = 1 + (32 + 2x1 - 3)/2= 16  
Khi stride (s) > 1 mỗi lần trượt cửa sổ sẽ nhảy s bước từ đó sẽ giảm được s lần trên ảnh gốc. Ví dụ 1234567 cửa sổ là 3 ô thì trượt từ cửa sổ đầu sang cửa sổ tiếp với s= 2 thì cửa sổ 123 tạo 1 đầu ra sang 345 là 1 ô đầu ra tiếp là 567 ô thứ 4 => đầu ra có 3 ô

## 3. Truyền xuôi qua phép tích chập (convolution)

Tại một vị trí $(i,j)$, mỗi bộ lọc lấy một vùng cục bộ của ảnh:

$$
X_{\text{patch}}
\in
\mathbb{R}^{C\times HH\times WW}.
$$

Sau đó tính tích vô hướng với bộ lọc và cộng bias:

$$
Y_{n,f,i,j}
=
\sum_{c,u,v}
X_{n,c,iS+u,jS+v}
W_{f,c,u,v}
+b_f.
$$

Trong phần đầu, **không cần tối ưu tốc độ**. Vòng lặp lồng nhau là chấp nhận được vì mục
tiêu là hiểu phép toán.

### Thực hành 1 — Cài đặt `conv_forward_naive`

In [16]:
def conv_forward_naive(
    x,
    w,
    b,
    conv_param,
):
    """
    x: (N, C, H, W)
    w: (F, C, HH, WW)
    b: (F,)
    conv_param:
        stride: int
        pad: int

    Trả về
    -------
    out: (N, F, H_out, W_out)
    cache: dữ liệu cần cho truyền ngược
    """
    stride = conv_param["stride"]
    pad = conv_param["pad"]

    # ============================================================
    # TODO: CODE CỦA BẠN
    #
    # 1. Tính H_out, W_out.
    # 2. Pad x theo hai chiều không gian.
    # 3. Trượt từng bộ lọc trên từng ảnh.
    # 4. Tính tích vô hướng và cộng bias.
    # ============================================================
    N, C, H, W = x.shape
    F, _, HH, WW = w.shape

    H_out = 1 + (H + 2 * pad - HH) // stride
    W_out = 1 + (W + 2 * pad - WW) // stride

    x_pad = np.pad(
        x,
        pad_width= ((0,0),(0,0),(pad,pad),(pad,pad)),
        mode= "constant",
        constant_values= 0
    )

    out = np.zeros((N, F, H_out, W_out), dtype=x.dtype)

    for n in range(N):
        for f in range(F):
            for i in range(H_out):
                h_start = i * stride
                h_end = h_start + HH
                for j in range(W_out):
                    w_start = j * stride
                    w_end = w_start + WW
                    x_slice = x_pad[n, :, h_start:h_end, w_start:w_end]
                    out[n, f, i, j] = np.sum(x_slice * w[f]) + b[f]

    cache = (x, w, b, conv_param)
    return out,cache
    # ============================================================
    # KẾT THÚC CODE CỦA BẠN
    # ============================================================

### Kiểm tra bằng PyTorch

Ta so sánh với `torch.nn.functional.conv2d`. Đây chỉ là phép kiểm tra kết quả; phần
cài đặt của bạn vẫn phải dùng NumPy.

In [17]:
N, C, H, W = 2, 3, 7, 7
F_out, HH, WW = 4, 3, 3

x = np.random.randn(N, C, H, W)
w = np.random.randn(F_out, C, HH, WW)
b = np.random.randn(F_out)

conv_param = {
    "stride": 2,
    "pad": 1,
}

out_np, cache = conv_forward_naive(
    x,
    w,
    b,
    conv_param,
)

out_torch = F.conv2d(
    torch.tensor(x, dtype=torch.float64),
    torch.tensor(w, dtype=torch.float64),
    torch.tensor(b, dtype=torch.float64),
    stride=2,
    padding=1,
).detach().numpy()

print("Kích thước:", out_np.shape)
print("Sai số tương đối:", relative_error(out_np, out_torch))

assert out_np.shape == out_torch.shape
assert relative_error(out_np, out_torch) < 1e-10

print("✓ conv_forward_naive đúng trên kiểm tra này")

Kích thước: (2, 4, 4, 4)
Sai số tương đối: 9.555728035481153e-17
✓ conv_forward_naive đúng trên kiểm tra này


## 4. Tích chập như một phép xử lý ảnh

Bộ lọc không nhất thiết bắt đầu bằng các trọng số ngẫu nhiên. Ta có thể tự đặt bộ lọc (kernel) để
thực hiện các phép xử lý ảnh cổ điển.

Ví dụ bộ lọc (kernel) phát hiện cạnh theo chiều ngang:

$$
K=
\begin{bmatrix}
-1 & -1 & -1\\
0 & 0 & 0\\
1 & 1 & 1
\end{bmatrix}.
$$

CNN học các bộ lọc (kernel) tương tự từ dữ liệu thay vì ta phải thiết kế bằng tay.

### Thực hành quan sát — Áp dụng bộ lọc lên ảnh CIFAR-10

In [18]:
from torchvision import datasets, transforms

raw_cifar = datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=transforms.ToTensor(),
)

image, label = raw_cifar[0]
image_np = image.numpy()[None, ...]

edge = np.array([
    [-1.0, -1.0, -1.0],
    [ 0.0,  0.0,  0.0],
    [ 1.0,  1.0,  1.0],
])

# Cùng kernel trên ba kênh màu.
edge_filter = np.tile(
    edge[None, None, :, :],
    (1, 3, 1, 1),
) / 3.0

edge_bias = np.zeros(1)

edge_out, _ = conv_forward_naive(
    image_np,
    edge_filter,
    edge_bias,
    {"stride": 1, "pad": 1},
)

fig = plt.figure(figsize=(8, 3))

ax = fig.add_subplot(1, 2, 1)
ax.imshow(
    np.transpose(
        image_np[0],
        (1, 2, 0),
    )
)
ax.set_title("Ảnh gốc")
ax.axis("off")

ax = fig.add_subplot(1, 2, 2)
ax.imshow(
    edge_out[0, 0],
    cmap="gray",
)
ax.set_title("Đáp ứng bộ lọc cạnh")
ax.axis("off")

plt.show()

  1%|▏         | 2.29M/170M [00:33<41:11, 68.1kB/s]


KeyboardInterrupt: 

### Câu hỏi 2 — Chia sẻ tham số

Giả sử cùng một cạnh xuất hiện ở góc trái và góc phải của ảnh.

Vì sao cơ chế chia sẻ tham số của CNN giúp cùng một bộ lọc có thể phản ứng với cả hai
vị trí, trong khi một lớp kết nối đầy đủ không có tính chất này một cách tự nhiên?

### ✍️ Câu trả lời của bạn

<!-- Viết câu trả lời bên dưới. Không xóa ô này. -->

## 5. Truyền ngược qua phép tích chập

Giả sử ta nhận đạo hàm từ lớp phía sau:

$$
dY
=
\frac{\partial L}{\partial Y}.
$$

Cần tính:

$$
dX,\qquad dW,\qquad db.
$$

Trực giác:

- $db_f$ là tổng các đạo hàm đi qua mọi vị trí của bản đồ đặc trưng (feature map) $f$;
- $dW_f$ cộng đóng góp của mọi patch đã dùng bộ lọc $f$;
- $dX$ cộng đóng góp từ tất cả các bộ lọc và tất cả cửa sổ chứa vị trí đó.

Điểm khó nhất là **một pixel có thể thuộc nhiều cửa sổ**, vì vậy các đóng góp vào $dX$
phải được cộng dồn.

### Thực hành 2 — Cài đặt `conv_backward_naive`

In [ ]:
def conv_backward_naive(
    dout,
    cache,
):
    """
    dout: đạo hàm theo đầu ra của convolution.

    Trả về
    -------
    dx, dw, db
    """
    # ============================================================
    # TODO: CODE CỦA BẠN
    #
    # Dùng lại thông tin trong cache của conv_forward_naive.
    # Nhớ cộng dồn gradient ở các vùng chồng lấn.
    # ============================================================
    raise NotImplementedError
    # ============================================================
    # KẾT THÚC CODE CỦA BẠN
    # ============================================================

### Kiểm tra truyền ngược bằng vi phân tự động của PyTorch (autograd)

In [ ]:
x_t = torch.tensor(
    x,
    dtype=torch.float64,
    requires_grad=True,
)
w_t = torch.tensor(
    w,
    dtype=torch.float64,
    requires_grad=True,
)
b_t = torch.tensor(
    b,
    dtype=torch.float64,
    requires_grad=True,
)

torch_out = F.conv2d(
    x_t,
    w_t,
    b_t,
    stride=2,
    padding=1,
)

dout = np.random.randn(
    *torch_out.shape
)

torch_out.backward(
    torch.tensor(
        dout,
        dtype=torch.float64,
    )
)

out_np, cache = conv_forward_naive(
    x,
    w,
    b,
    conv_param,
)

dx, dw, db = conv_backward_naive(
    dout,
    cache,
)

print("dx error:", relative_error(dx, x_t.grad.numpy()))
print("dw error:", relative_error(dw, w_t.grad.numpy()))
print("db error:", relative_error(db, b_t.grad.numpy()))

assert relative_error(dx, x_t.grad.numpy()) < 1e-8
assert relative_error(dw, w_t.grad.numpy()) < 1e-8
assert relative_error(db, b_t.grad.numpy()) < 1e-10

print("✓ conv_backward_naive đúng trên kiểm tra này")

## 6. Gộp cực đại (max pooling)

Max pooling giảm kích thước không gian bằng cách giữ giá trị lớn nhất trong mỗi cửa sổ.

Ví dụ với cửa sổ $2\times2$:

$$
\begin{bmatrix}
1 & 5\\
2 & 3
\end{bmatrix}
\rightarrow 5.
$$

Pooling không có tham số học được.

Trong truyền ngược, gradient chỉ được truyền về vị trí đã tạo ra giá trị cực đại.

### Thực hành 3 — `max_pool_forward_naive`

In [ ]:
def max_pool_forward_naive(
    x,
    pool_param,
):
    """
    pool_param:
        pool_height
        pool_width
        stride
    """
    # ============================================================
    # TODO: CODE CỦA BẠN
    # ============================================================
    raise NotImplementedError
    # ============================================================
    # KẾT THÚC CODE CỦA BẠN
    # ============================================================

In [ ]:
pool_x = np.random.randn(
    2,
    3,
    8,
    8,
)

pool_param = {
    "pool_height": 2,
    "pool_width": 2,
    "stride": 2,
}

pool_np, pool_cache = max_pool_forward_naive(
    pool_x,
    pool_param,
)

pool_torch = F.max_pool2d(
    torch.tensor(
        pool_x,
        dtype=torch.float64,
    ),
    kernel_size=2,
    stride=2,
).numpy()

print("Sai số:", relative_error(pool_np, pool_torch))

assert relative_error(pool_np, pool_torch) < 1e-12
print("✓ max_pool_forward_naive đúng")

### Thực hành 4 — `max_pool_backward_naive`

In [ ]:
def max_pool_backward_naive(
    dout,
    cache,
):
    # ============================================================
    # TODO: CODE CỦA BẠN
    #
    # Gradient phải quay về vị trí cực đại trong từng cửa sổ.
    # ============================================================
    raise NotImplementedError
    # ============================================================
    # KẾT THÚC CODE CỦA BẠN
    # ============================================================

In [ ]:
pool_x_t = torch.tensor(
    pool_x,
    dtype=torch.float64,
    requires_grad=True,
)

pool_out_t = F.max_pool2d(
    pool_x_t,
    kernel_size=2,
    stride=2,
)

pool_dout = np.random.randn(
    *pool_out_t.shape
)

pool_out_t.backward(
    torch.tensor(
        pool_dout,
        dtype=torch.float64,
    )
)

pool_np, pool_cache = max_pool_forward_naive(
    pool_x,
    pool_param,
)

pool_dx = max_pool_backward_naive(
    pool_dout,
    pool_cache,
)

print(
    "dx error:",
    relative_error(
        pool_dx,
        pool_x_t.grad.numpy(),
    )
)

assert relative_error(
    pool_dx,
    pool_x_t.grad.numpy(),
) < 1e-10

print("✓ max_pool_backward_naive đúng")

### Câu hỏi 3 — Truyền ngược qua max pooling

Vì sao gradient của max pooling chỉ đi qua vị trí cực đại?

Nếu trong một cửa sổ có hai giá trị cực đại bằng nhau, implementation của bạn xử lý
trường hợp này như thế nào?

### ✍️ Câu trả lời của bạn

<!-- Viết câu trả lời bên dưới. Không xóa ô này. -->

## 7. Từ phép toán đơn lẻ đến CNN

Một CNN thực tế thường ghép các lớp thành khối:

$$
\text{Conv}
\rightarrow
\text{ReLU}
\rightarrow
\text{Pool}.
$$

Sau nhiều khối, bản đồ đặc trưng (bản đồ đặc trưng (feature map)s) được tổng hợp thành vector đặc trưng và đưa vào lớp phân loại.

Ở Q3, PyTorch sẽ xử lý nhanh các phép toán này. Nhưng khi gọi:

```python
nn.Conv2d(...)
```

bạn nên hình dung bên dưới đang thực hiện đúng các phép tính vừa cài đặt.

### Câu hỏi 4 — Tích chập so với lớp kết nối đầy đủ

Nêu **hai lợi thế** của convolution so với kết nối đầy đủ (fully connected) khi làm việc với ảnh, và
**một hạn chế** của việc dùng bộ lọc (kernel) quá nhỏ nếu mạng quá nông.

### ✍️ Câu trả lời của bạn

<!-- Viết câu trả lời bên dưới. Không xóa ô này. -->

## Tóm tắt Q1

Bạn đã tự cài đặt:

- convolution truyền xuôi;
- convolution truyền ngược;
- max pooling truyền xuôi;
- max pooling truyền ngược.

Các phép toán này là nền tảng để xây CNN ở Q3.